# PCG-MAS — Frontier Cloud Runner (16-cell scaffold, 2-cell execution)

Full 8-dataset x 2-frontier-model grid (16 cells) wired for PCG-MAS R1-R5 + additional
paper artifacts + all 6 SOTA baselines. A budget gate executes only the 2 manuscript-critical
cells; the other 14 are scaffolded and dry-runnable but never spent on.

- **Datasets (8):** fever, hotpotqa, twowiki, toolbench, pubmedqa, tatqa, weblinx, synthetic
- **Frontier models (2):** Llama-3.3-70B (quantized local), deepseek-v3 (API-only)
- **Execution targets (2):** `toolbench:Llama-3.3-70B`, `weblinx:deepseek-v3`

Secrets via Colab Secrets only (HF_TOKEN, ANTHROPIC_API_KEY). Never paste tokens inline.


## 1 - Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/pcg-submission'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


Mounted at /content/drive
Drive root: /content/drive/MyDrive/pcg-submission


In [ ]:
!nvidia-smi

Fri May 29 07:40:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2 - Clone / update repo


In [ ]:
# %cd /content
# ![ -d pcg ] || git clone https://github.com/anonymous-submission/proof-carrying-multi-agents.git pcg
# %cd /content/pcg
# !git pull --ff-only || true
# !git log --oneline -1


In [ ]:
from pathlib import Path
import os

REPO = Path("/content/drive/MyDrive/pcg-submission")
RUN_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive")

print("REPO exists:", REPO.exists())
print("pyproject:", (REPO / "pyproject.toml").exists())
print("requirements:", (REPO / "requirements.txt").exists())
print("src/pcg:", (REPO / "src" / "pcg").exists())

assert REPO.exists(), "Repo path missing"
assert (REPO / "pyproject.toml").exists(), "pyproject.toml missing; wrong working directory"
assert (REPO / "src" / "pcg").exists(), "src/pcg missing; wrong repo copy"

os.chdir(REPO)
print("cwd:", os.getcwd())

REPO exists: True
pyproject: True
requirements: True
src/pcg: True
cwd: /content/drive/MyDrive/pcg-submission


In [ ]:
from pathlib import Path
import os

RUN_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive")
HF_CACHE = RUN_ROOT / "hf_cache"
RESULTS_DRIVE = RUN_ROOT / "results"
ZIPS = RUN_ROOT / "zips"

for p in [HF_CACHE, RESULTS_DRIVE, ZIPS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

print("HF cache:", HF_CACHE)
print("results:", RESULTS_DRIVE)

HF cache: /content/drive/MyDrive/pcg-submission-drive/hf_cache
results: /content/drive/MyDrive/pcg-submission-drive/results


In [ ]:
# %%bash
# set -e

# cd /content/drive/MyDrive/pcg-submission

# rm -rf results
# ln -s /content/drive/MyDrive/pcg-submission-drive/results results

# echo "cwd:"
# pwd
# echo "results symlink:"
# ls -l results

## 3 - Secrets (Colab Secrets only)
Set `HF_TOKEN` and `ANTHROPIC_API_KEY` in the Colab Secrets panel (key icon, left).
HF_TOKEN must have access to gated meta-llama/Llama-3.3-70B-Instruct.


In [ ]:
import os
from google.colab import userdata

# Standard secrets
for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DEEPSEEK_API_KEY"):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception as e:
        print(f"{k}: not set in Secrets ({e})")

# Hugging Face token priority:
# 1. HF_INFERENCE if present, because this is the inference-provider-enabled token
# 2. otherwise HF_TOKEN
hf_value = None

try:
    hf_value = userdata.get("HF_INFERENCE")
    if hf_value:
        os.environ["HF_TOKEN"] = hf_value
        os.environ["HF_INFERENCE"] = hf_value
        print("HF_INFERENCE: loaded and mapped to HF_TOKEN")
except Exception as e:
    print(f"HF_INFERENCE: not set in Secrets ({e})")

if not hf_value:
    try:
        hf_value = userdata.get("HF_TOKEN")
        if hf_value:
            os.environ["HF_TOKEN"] = hf_value
            print("HF_TOKEN: loaded")
    except Exception as e:
        print(f"HF_TOKEN: not set in Secrets ({e})")

print("HF_TOKEN         :", "set" if os.environ.get("HF_TOKEN") else "MISSING")
print("HF_INFERENCE    :", "set" if os.environ.get("HF_INFERENCE") else "MISSING")
print("OPENAI_API_KEY   :", "set" if os.environ.get("OPENAI_API_KEY") else "MISSING")
print("ANTHROPIC_API_KEY:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")
print("DEEPSEEK_API_KEY :", "set" if os.environ.get("DEEPSEEK_API_KEY") else "MISSING")


HF_INFERENCE: loaded and mapped to HF_TOKEN
HF_TOKEN         : set
HF_INFERENCE    : set
OPENAI_API_KEY   : set
ANTHROPIC_API_KEY: set
DEEPSEEK_API_KEY : set


## 4 - HF cache on Drive (download weights once)


In [ ]:
import os
HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE}/datasets'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE}/transformers'
print('HF_HOME =', os.environ['HF_HOME'])


HF_HOME = /content/drive/MyDrive/pcg-submission/hf_cache


## 5 - Install pinned deps (matches local multi-agents venv)
Same pinset as local: transformers 4.46.x, torch 2.5.x, tokenizers <0.21, hf_hub <1.0.
Plus GPU-only inference extras (bitsandbytes for 4-bit, vllm if the GPU supports it).


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/drive/MyDrive/pcg-submission")
os.chdir(REPO)

BASE = [
    "transformers>=4.45,<4.50",
    "tokenizers>=0.20,<0.21",
    "huggingface_hub>=0.23,<1.0",
    "accelerate>=1.0,<2.0",
    "safetensors>=0.4,<1.0",
    "datasets>=2.20,<5",
    "sentence-transformers>=3.0,<6",
    "rank-bm25>=0.2",
    "faiss-cpu>=1.7",
    "numpy>=1.26,<3",
    "scipy>=1.11,<2",
    "scikit-learn>=1.3,<2",
    "pandas>=2.0,<4",
    "tiktoken>=0.7",
    "openai>=1.30",
    "anthropic>=0.40",
    "langgraph>=0.2,<2",
    "langchain-core>=0.3,<2",
    "matplotlib",
    "seaborn",
    "plotly",
    "loguru",
    "tqdm",
    "pyyaml",
    "bitsandbytes>=0.43",
]

def pip_install(args):
    print("RUN:", " ".join(args))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

pip_install(BASE)

# Install the repo itself.
pip_install(["-e", "."])

# Keep torch line separate and explicit.
pip_install(["torch>=2.4,<2.6", "torchvision==0.20.1"])

print("Install complete")

RUN: transformers>=4.45,<4.50 tokenizers>=0.20,<0.21 huggingface_hub>=0.23,<1.0 accelerate>=1.0,<2.0 safetensors>=0.4,<1.0 datasets>=2.20,<5 sentence-transformers>=3.0,<6 rank-bm25>=0.2 faiss-cpu>=1.7 numpy>=1.26,<3 scipy>=1.11,<2 scikit-learn>=1.3,<2 pandas>=2.0,<4 tiktoken>=0.7 openai>=1.30 anthropic>=0.40 langgraph>=0.2,<2 langchain-core>=0.3,<2 matplotlib seaborn plotly loguru tqdm pyyaml bitsandbytes>=0.43
RUN: -e .
RUN: torch>=2.4,<2.6 torchvision==0.20.1
Install complete


In [ ]:
import torch, transformers, tokenizers, huggingface_hub, numpy as np

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("numpy:", np.__version__)

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


torch: 2.5.1+cu124
cuda: True
transformers: 4.46.3
tokenizers: 0.20.3
huggingface_hub: 0.36.2
numpy: 2.0.2


## verify PCG-MAS imports

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/drive/MyDrive/pcg-submission")
os.chdir(REPO)

print("cwd:", os.getcwd())
print("python:", sys.executable)
print("pyproject exists:", (REPO / "pyproject.toml").exists())

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-e", "."
])

print("editable install complete")

cwd: /content/drive/MyDrive/pcg-submission
python: /usr/bin/python3
pyproject exists: True
editable install complete


In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/drive/MyDrive/pcg-submission")
SRC = REPO / "src"

# Fallback: make src importable even if editable install metadata name is odd.
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pcg
from pcg.orchestrator.langgraph_flow import OrchestratorConfig

print("PCG-MAS import: OK")
print("pcg:", pcg)
print("OrchestratorConfig:", OrchestratorConfig)

PCG-MAS import: OK
pcg: <module 'pcg' from '/content/drive/MyDrive/pcg-submission/src/pcg/__init__.py'>
OrchestratorConfig: <class 'pcg.orchestrator.langgraph_flow.OrchestratorConfig'>


## 6 - GPU check + adaptive 70B backend selection
Picks vLLM if a big GPU is present, else 4-bit NF4 local, else API fallback.


In [ ]:
import torch, subprocess
has_cuda = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_cuda else 'none'
gpu_mem_gb = (torch.cuda.get_device_properties(0).total_memory/1e9) if has_cuda else 0
print(f'CUDA={has_cuda}  GPU={gpu_name}  mem={gpu_mem_gb:.0f}GB')

# Decide 70B strategy by available VRAM
if gpu_mem_gb >= 75:
    LLAMA70B_MODE = 'vllm'        # A100 80GB / H100: full or AWQ via vLLM
elif gpu_mem_gb >= 38:
    LLAMA70B_MODE = 'nf4_4bit'    # 40GB A100: 4-bit NF4 local
else:
    LLAMA70B_MODE = 'api'         # fallback: HF Inference API
print('Llama-3.3-70B backend mode:', LLAMA70B_MODE)
print('deepseek-v3 backend mode  : api (hf_inference, always)')


CUDA=True  GPU=NVIDIA A100-SXM4-80GB  mem=85GB
Llama-3.3-70B backend mode: vllm
deepseek-v3 backend mode  : api (hf_inference, always)


## 7 - The 16-cell grid + budget gate
Full scaffold defined; only ALLOWLIST cells actually execute.


In [ ]:
DATASETS = ['fever','hotpotqa','twowiki','toolbench','pubmedqa','tatqa','weblinx','synthetic']
FRONTIER_MODELS = ['Llama-3.3-70B','deepseek-v3']

FULL_GRID = [f'{d}:{m}' for m in FRONTIER_MODELS for d in DATASETS]  # 16 cells
assert len(FULL_GRID) == 16

# Budget gate: only these execute. Everything else is scaffolded, never spent on.
ALLOWLIST = ['toolbench:Llama-3.3-70B', 'weblinx:deepseek-v3']

print(f'Full grid: {len(FULL_GRID)} cells')
for c in FULL_GRID:
    flag = 'EXECUTE' if c in ALLOWLIST else 'scaffold'
    print(f'  [{flag:8s}] {c}')


Full grid: 16 cells
  [scaffold] fever:Llama-3.3-70B
  [scaffold] hotpotqa:Llama-3.3-70B
  [scaffold] twowiki:Llama-3.3-70B
  [EXECUTE ] toolbench:Llama-3.3-70B
  [scaffold] pubmedqa:Llama-3.3-70B
  [scaffold] tatqa:Llama-3.3-70B
  [scaffold] weblinx:Llama-3.3-70B
  [scaffold] synthetic:Llama-3.3-70B
  [scaffold] fever:deepseek-v3
  [scaffold] hotpotqa:deepseek-v3
  [scaffold] twowiki:deepseek-v3
  [scaffold] toolbench:deepseek-v3
  [scaffold] pubmedqa:deepseek-v3
  [scaffold] tatqa:deepseek-v3
  [EXECUTE ] weblinx:deepseek-v3
  [scaffold] synthetic:deepseek-v3


## 8 - Output tree on Drive (matches local dataset:model layout)
Results land under Drive in the same structure local expects, for clean sync-back.


In [ ]:
%%bash
set -e

DRIVE_REPO="/content/drive/MyDrive/pcg-submission"
RUN_ROOT="/content/drive/MyDrive/pcg-submission-drive"
LOCAL_REPO="/content/pcg"
OUT_ROOT="$RUN_ROOT/results"

echo "Drive repo: $DRIVE_REPO"
echo "Local repo: $LOCAL_REPO"
echo "Out root  : $OUT_ROOT"

test -f "$DRIVE_REPO/pyproject.toml"
test -d "$DRIVE_REPO/src/pcg"

mkdir -p "$OUT_ROOT/tables/csv/experiment_json"
mkdir -p "$OUT_ROOT/tables/csv/ablations_outputs"
mkdir -p "$OUT_ROOT/baselines"
mkdir -p "$OUT_ROOT/figures"
mkdir -p "$OUT_ROOT/audit"
mkdir -p "$RUN_ROOT/zips"
mkdir -p "$RUN_ROOT/hf_cache"

rm -rf "$LOCAL_REPO"
mkdir -p "$LOCAL_REPO"

# Copy code from Drive to fast local Colab disk.
# Exclude results because results will be a symlink to Drive output.
rsync -a \
  --exclude "results" \
  --exclude ".git" \
  --exclude ".venvs" \
  --exclude "__pycache__" \
  --exclude "*.pyc" \
  "$DRIVE_REPO/" "$LOCAL_REPO/"

# Symlink works here because /content is local Linux filesystem.
rm -rf "$LOCAL_REPO/results"
ln -s "$OUT_ROOT" "$LOCAL_REPO/results"

echo
echo "Local repo ready:"
ls -lh "$LOCAL_REPO" | head

echo
echo "results symlink:"
ls -l "$LOCAL_REPO/results"

Drive repo: /content/drive/MyDrive/pcg-submission
Local repo: /content/pcg
Out root  : /content/drive/MyDrive/pcg-submission-drive/results

Local repo ready:
total 180K
drwx------  5 root root 4.0K May 22 07:18 app
drwx------  3 root root 4.0K May 28 15:55 artifacts
drwx------  2 root root 4.0K May 28 15:55 configs
drwx------  2 root root 4.0K May 20 07:34 docs
drwx------  4 root root 4.0K May 28 19:39 hf_cache
drwx------  2 root root 4.0K Apr 28 20:18 latex
-rw-------  1 root root 1.4K May  9 19:14 Makefile
-rw-------  1 root root 2.9K May 27 02:16 pyproject.toml
-rw-------  1 root root  21K May 22 22:55 README_creator_v4.txt

results symlink:
lrwxrwxrwx 1 root root 51 May 29 07:47 /content/pcg/results -> /content/drive/MyDrive/pcg-submission-drive/results


In [ ]:
import os, sys, subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
RUN_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive")
HF_CACHE = RUN_ROOT / "hf_cache"

os.chdir(LOCAL_REPO)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

# Ensure imports use local repo, not stale Drive path.
SRC = LOCAL_REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

print("cwd:", os.getcwd())
print("HF_HOME:", os.environ["HF_HOME"])
print("results target:", os.readlink("results"))

cwd: /content/pcg
HF_HOME: /content/drive/MyDrive/pcg-submission-drive/hf_cache
results target: /content/drive/MyDrive/pcg-submission-drive/results


In [ ]:
import pcg
from pcg.orchestrator.langgraph_flow import OrchestratorConfig

print("PCG-MAS import: OK")
print("pcg module:", pcg)
print("OrchestratorConfig:", OrchestratorConfig)

PCG-MAS import: OK
pcg module: <module 'pcg' from '/content/drive/MyDrive/pcg-submission/src/pcg/__init__.py'>
OrchestratorConfig: <class 'pcg.orchestrator.langgraph_flow.OrchestratorConfig'>


In [ ]:
import os, sys, subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
LOCAL_SRC = LOCAL_REPO / "src"
DRIVE_SRC = "/content/drive/MyDrive/pcg-submission/src"

os.chdir(LOCAL_REPO)

# Remove old Drive source path if it was inserted earlier
sys.path = [p for p in sys.path if p != DRIVE_SRC]

# Put local source first
if str(LOCAL_SRC) in sys.path:
    sys.path.remove(str(LOCAL_SRC))
sys.path.insert(0, str(LOCAL_SRC))

# Reinstall editable from local repo
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(LOCAL_REPO)])

# Clear old module if already imported
for m in list(sys.modules):
    if m == "pcg" or m.startswith("pcg."):
        del sys.modules[m]

import pcg
from pcg.orchestrator.langgraph_flow import OrchestratorConfig

print("cwd:", os.getcwd())
print("pcg module:", pcg)
print("OrchestratorConfig:", OrchestratorConfig)
print("results target:", os.readlink(LOCAL_REPO / "results"))

cwd: /content/pcg
pcg module: <module 'pcg' from '/content/pcg/src/pcg/__init__.py'>
OrchestratorConfig: <class 'pcg.orchestrator.langgraph_flow.OrchestratorConfig'>
results target: /content/drive/MyDrive/pcg-submission-drive/results


## force local /content/pcg execution

In [ ]:
import os, sys, subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
RUN_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive")
OUT_ROOT = RUN_ROOT / "results"
HF_CACHE = RUN_ROOT / "hf_cache"
LOG_ROOT = RUN_ROOT / "logs"
ZIP_ROOT = RUN_ROOT / "zips"

for p in [OUT_ROOT, HF_CACHE, LOG_ROOT, ZIP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE)

os.chdir(LOCAL_REPO)

# Make sure local source wins over Drive source.
DRIVE_SRC = "/content/drive/MyDrive/pcg-submission/src"
LOCAL_SRC = str(LOCAL_REPO / "src")
sys.path = [p for p in sys.path if p != DRIVE_SRC]
if LOCAL_SRC in sys.path:
    sys.path.remove(LOCAL_SRC)
sys.path.insert(0, LOCAL_SRC)

# Reinstall editable from local repo.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(LOCAL_REPO)])

# Clear any stale pcg module from previous Drive import.
for m in list(sys.modules):
    if m == "pcg" or m.startswith("pcg."):
        del sys.modules[m]

import pcg
from pcg.orchestrator.langgraph_flow import OrchestratorConfig

print("cwd:", os.getcwd())
print("pcg module:", pcg)
print("results target:", os.readlink(LOCAL_REPO / "results"))
print("HF_HOME:", os.environ["HF_HOME"])

assert str(pcg.__file__).startswith("/content/pcg/src/pcg"), pcg.__file__
assert os.readlink(LOCAL_REPO / "results") == str(OUT_ROOT)

cwd: /content/pcg
pcg module: <module 'pcg' from '/content/pcg/src/pcg/__init__.py'>
results target: /content/drive/MyDrive/pcg-submission-drive/results
HF_HOME: /content/drive/MyDrive/pcg-submission-drive/hf_cache


## 9 - Backend env for the runners
The PCG runners read backend.kind from config; we override per-cell via env the
runner respects. For 70B local we use hf_local; for deepseek-v3 we use hf_inference.


In [ ]:
import os, subprocess, textwrap
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
RUN_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive")
OUT_ROOT = RUN_ROOT / "results"
LOG_ROOT = RUN_ROOT / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

# Two heavy allowlist cells.
ALLOWLIST = [
    "toolbench:Llama-3.3-70B",
    "weblinx:deepseek-v3",
]

# GPU-adaptive 70B mode.
def _gpu_mem_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        return int(out) / 1024
    except Exception:
        return 0.0

GPU_MEM_GB = _gpu_mem_gb()

if GPU_MEM_GB >= 75:
    LLAMA70B_MODE = "vllm"
elif GPU_MEM_GB >= 38:
    LLAMA70B_MODE = "hf_local"
else:
    LLAMA70B_MODE = "api"

def pcg_backend_for(cell: str) -> str:
    model = cell.split(":", 1)[1]
    if model == "deepseek-v3":
        return "deepseek"
    if model == "Llama-3.3-70B":
        return "hf_local" if LLAMA70B_MODE in {"hf_local", "vllm"} else "hf_inference"
    return "hf_local"

def script_supports_arg(path: str, arg: str) -> bool:
    try:
        out = subprocess.run(
            ["bash", "-lc", f"cd {LOCAL_REPO} && python {path} --help"],
            text=True,
            capture_output=True,
            timeout=60,
        )
        return arg in (out.stdout + out.stderr)
    except Exception:
        return False

RUNNER = "scripts/runs/run_additional_paper_artifacts.sh"
RUNNER_SUPPORTS_BACKEND = "--backend" in Path(LOCAL_REPO / RUNNER).read_text()

print("GPU_MEM_GB:", round(GPU_MEM_GB, 1))
print("LLAMA70B_MODE:", LLAMA70B_MODE)
print("RUNNER_SUPPORTS_BACKEND:", RUNNER_SUPPORTS_BACKEND)
print("OUT_ROOT:", OUT_ROOT)

for c in ALLOWLIST:
    print(f"{c:32s} -> {pcg_backend_for(c)}")

assert RUNNER_SUPPORTS_BACKEND, (
    "run_additional_paper_artifacts.sh does not support --backend. "
    "Patch the runner before continuing, otherwise DeepSeek/70B backend selection may be wrong."
)

def backend_arg(backend: str) -> str:
    return f"--backend {backend}" if RUNNER_SUPPORTS_BACKEND else ""

def run_logged(name: str, cmd: str, tail: int = 80):
    log_path = LOG_ROOT / f"{name}.log"
    full = f"""
    set -euo pipefail
    cd {LOCAL_REPO}
    echo "LOG: {log_path}"
    {cmd} 2>&1 | tee {log_path} | tail -{tail}
    """
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)
    print(cmd)
    subprocess.check_call(["bash", "-lc", textwrap.dedent(full)])


GPU_MEM_GB: 80.0
LLAMA70B_MODE: vllm
RUNNER_SUPPORTS_BACKEND: True
OUT_ROOT: /content/drive/MyDrive/pcg-submission-drive/results
toolbench:Llama-3.3-70B          -> hf_local
weblinx:deepseek-v3              -> deepseek


## 10 - PREFLIGHT: dry n=3 on one allowlist cell
Smoke the full additional-artifacts pipeline at tiny n before spending budget.


In [ ]:
from pathlib import Path

LOG_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive/logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("LOG_ROOT exists:", LOG_ROOT.exists())
print("LOG_ROOT:", LOG_ROOT)

LOG_ROOT exists: True
LOG_ROOT: /content/drive/MyDrive/pcg-submission-drive/logs


In [ ]:
import subprocess, os
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
LOG_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive/logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

cmd = """
cd /content/pcg
PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
  --cells toolbench:Llama-3.3-70B \
  --n-examples 3 \
  --seed 0 \
  --backend hf_local
"""

log = LOG_ROOT / "preflight_llama70b_direct_n3.log"

print("running...")
print(cmd)
res = subprocess.run(
    ["bash", "-lc", cmd],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

log.write_text(res.stdout)

print("returncode:", res.returncode)
print("log:", log)
print("\n===== LAST 120 LINES =====")
print("\n".join(res.stdout.splitlines()[-120:]))

running...

cd /content/pcg
PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh   --cells toolbench:Llama-3.3-70B   --n-examples 3   --seed 0   --backend hf_local

returncode: 1
log: /content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_direct_n3.log

===== LAST 120 LINES =====
scripts/runs/run_additional_paper_artifacts.sh: line 46: .venvs/multi-agents/bin/activate: No such file or directory


In [ ]:
from pathlib import Path

p = Path("/content/pcg/scripts/runs/run_additional_paper_artifacts.sh")
s = p.read_text()

old = 'source .venvs/multi-agents/bin/activate'
new = '''
# In Colab, dependencies are installed into the active runtime.
# Locally, activate the repo venv only if it exists.
if [ -f ".venvs/multi-agents/bin/activate" ]; then
  source .venvs/multi-agents/bin/activate
fi
'''.strip()

if old in s:
    s = s.replace(old, new)
    p.write_text(s)
    print("patched venv activation guard")
else:
    print("activation line not found; showing possible activate lines:")
    for i, line in enumerate(s.splitlines(), 1):
        if "activate" in line or ".venvs" in line:
            print(i, line)

patched venv activation guard


In [ ]:
ALLOWLIST = [
    "toolbench:Llama-3.3-70B",
    "weblinx:deepseek-v3",
]

def pcg_backend_for(cell):
    model = cell.split(":", 1)[1]
    if model == "deepseek-v3":
        return "deepseek"
    if model == "Llama-3.3-70B":
        return "hf_inference"   # API fallback only; no local 70B on Colab
    return "hf_local"

for c in ALLOWLIST:
    print(f"{c:32s} -> {pcg_backend_for(c)}")

toolbench:Llama-3.3-70B          -> hf_inference
weblinx:deepseek-v3              -> deepseek


In [ ]:
import os, subprocess
from pathlib import Path

assert os.environ.get("HF_TOKEN"), "HF_TOKEN missing."

LOCAL_REPO = Path("/content/pcg")
LOG_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive/logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

cmd = """
cd /content/pcg
PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
  --cells toolbench:Llama-3.3-70B \
  --n-examples 1 \
  --seed 0 \
  --backend hf_inference
"""

log = LOG_ROOT / "preflight_llama70b_api_direct_n1_retry.log"

res = subprocess.run(
    ["bash", "-lc", cmd],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

log.write_text(res.stdout)

print("returncode:", res.returncode)
print("log:", log)
print("\n===== LAST 120 LINES =====")
print("\n".join(res.stdout.splitlines()[-120:]))

returncode: 0
log: /content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_api_direct_n1_retry.log

===== LAST 120 LINES =====

=== r4 privacy (measured): toolbench:Llama-3.3-70B ===
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 403, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/groq/openai/v1/chat/completions

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/pcg/src/pcg/backends/hf_inference.py", line 73, in generate
    response = self._client.chat_completion(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub

In [ ]:
import os, subprocess
from pathlib import Path

assert os.environ.get("DEEPSEEK_API_KEY"), "DEEPSEEK_API_KEY missing. Set/load it before deepseek."

LOCAL_REPO = Path("/content/pcg")
LOG_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive/logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

cmd = """
cd /content/pcg
PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
  --cells weblinx:deepseek-v3 \
  --n-examples 30 \
  --seed 0 \
  --backend deepseek
"""

log = LOG_ROOT / "preflight_deepseek_direct_n3.log"

res = subprocess.run(
    ["bash", "-lc", cmd],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

log.write_text(res.stdout)

print("returncode:", res.returncode)
print("log:", log)
print("\n===== LAST 120 LINES =====")
print("\n".join(res.stdout.splitlines()[-120:]))

returncode: 0
log: /content/drive/MyDrive/pcg-submission-drive/logs/preflight_deepseek_direct_n3.log

===== LAST 120 LINES =====
  variant=minus_v_gamma  cond=adv   harm=0.100  (  0.2s)
  variant=minus_v_entail cond=clean harm=0.200  (  0.1s)
  variant=minus_v_entail cond=adv   harm=0.000  (  0.2s)
wrote results/tables/csv/ablations_outputs/20260529-080908_ablations/weblinx__deepseek-v3.jsonl
wrote results/tables/csv/ablations_outputs/20260529-080908_ablations/weblinx__deepseek-v3__summary.json
wrote results/tables/channel_ablation.csv
wrote results/tables/channel_ablation.tex

ALL CELLS COMPLETE — out_dir: results/tables/csv/ablations_outputs/20260529-080908_ablations
cell                                     |        full(cl) |        full(ad) |   minus_v_h(cl) |   minus_v_h(ad) |  minus_v_pi(cl) |  minus_v_pi(ad) | minus_v_gam(cl) | minus_v_gam(ad) | minus_v_ent(cl) | minus_v_ent(ad)
   weblinx:deepseek-v3            |        0.167           0.000           0.167           0.000     

In [ ]:
import subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pcg")
LOG_ROOT = Path("/content/drive/MyDrive/pcg-submission-drive/logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

cmd = """
cd /content/pcg
PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
  --cells toolbench:Llama-3.3-70B \
  --n-examples 3 \
  --seed 0 \
  --backend hf_local
"""

log = LOG_ROOT / "preflight_llama70b_direct_n3.log"

res = subprocess.run(
    ["bash", "-lc", cmd],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

log.write_text(res.stdout)

print("returncode:", res.returncode)
print("log:", log)
print("\n===== LAST 120 LINES =====")
print("\n".join(res.stdout.splitlines()[-120:]))

returncode: 0
log: /content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_direct_n3.log

===== LAST 120 LINES =====

=== r5 scaling: toolbench:Llama-3.3-70B ===
  k             MISSING (no r5_overhead outputs found)
  |S0| (measured) sweep over top_k = [2, 4, 8, 16]

Loading checkpoint shards: 100%|██████████| 30/30 [15:59<00:00, 32.00s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
    top_k=2 ex=0: ERROR OutOfMemoryError

In [ ]:
from pathlib import Path
import os, time

log = Path("/content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_direct_n3.log")
print("log exists:", log.exists())
if log.exists():
    print("log size:", log.stat().st_size, "bytes")
    print("modified:", time.ctime(log.stat().st_mtime))
    print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
else:
    print("No log yet because subprocess writes it only after completion in the current direct cell.")

log exists: True
log size: 43825 bytes
modified: Thu May 28 23:15:41 2026
wrote results/figures/r4_privacy_frontier.png
wrote results/tables/r5_scaling.csv
wrote results/tables/r5_scaling.tex
wrote results/figures/r5_scaling.pdf
wrote results/figures/r5_scaling.png

[8/8] 23:15:41 Validate outputs
=== tables produced ===
  OK  results/tables/audit_calibration_summary.csv
  OK  results/tables/audit_calibration_summary.tex
  OK  results/tables/ablations.csv
  OK  results/tables/ablations.tex
  OK  results/tables/channel_ablation.csv
  OK  results/tables/channel_ablation.tex
  OK  results/tables/replay_drift_covgap.csv
  OK  results/tables/replay_drift_covgap.tex
  OK  results/tables/r3_open_mixed.csv
  OK  results/tables/r3_open_mixed.tex
  OK  results/tables/r4_privacy.csv
  OK  results/tables/r4_privacy.tex
  OK  results/tables/r5_scaling.csv
  OK  results/tables/r5_scaling.tex

=== figures produced ===
  OK  results/figures/ablations.pdf
  OK  results/figures/ablations.png
  OK  resul

In [ ]:
CELL = "toolbench:Llama-3.3-70B"
BACKEND = pcg_backend_for(CELL)

print(f"Llama preflight: {CELL} backend={BACKEND}")

run_logged(
    "preflight_llama70b_additional_artifacts_n3",
    f"""
    PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
      --cells {CELL} \
      --n-examples 3 \
      --seed 0 \
      {backend_arg(BACKEND)}
    """,
    tail=100,
)

Llama preflight: toolbench:Llama-3.3-70B backend=hf_local

preflight_llama70b_additional_artifacts_n3

    PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh       --cells toolbench:Llama-3.3-70B       --n-examples 3       --seed 0       --backend hf_local
    


CalledProcessError: Command '['bash', '-lc', '\nset -euo pipefail\ncd /content/pcg\necho "LOG: /content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_additional_artifacts_n3.log"\n\nPYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh       --cells toolbench:Llama-3.3-70B       --n-examples 3       --seed 0       --backend hf_local\n 2>&1 | tee /content/drive/MyDrive/pcg-submission-drive/logs/preflight_llama70b_additional_artifacts_n3.log | tail -100\n']' returned non-zero exit status 1.

## 10b - PREFLIGHT: DeepSeek API canary (n=3)
Cheap canary for the `weblinx:deepseek-v3` cell. Verifies DEEPSEEK_API_KEY auth
and that the official DeepSeek API serves requests BEFORE the n=30 run. If this
errors (401 / quota / model id), STOP and fix the key — do not run cell 12+.


In [ ]:
import os

if not os.environ.get("DEEPSEEK_API_KEY"):
    raise RuntimeError("DEEPSEEK_API_KEY is missing. Set it in Colab Secrets before running DeepSeek.")

CELL = "weblinx:deepseek-v3"
BACKEND = pcg_backend_for(CELL)

print(f"DeepSeek canary: {CELL} backend={BACKEND}")

run_logged(
    "preflight_deepseek_additional_artifacts_n3",
    f"""
    PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
      --cells {CELL} \
      --n-examples 3 \
      --seed 0 \
      {backend_arg(BACKEND)}
    """,
    tail=100,
)

## 11 - EXECUTE: full PCG additional-artifacts on the 2 allowlist cells
Manuscript-grade n. Adjust N as budget allows.


In [ ]:
N = 30

for CELL in ALLOWLIST:
    BACKEND = pcg_backend_for(CELL)
    safe_name = CELL.replace(":", "__").replace("/", "_")
    print(f"\n===== PCG additional artifacts: {CELL} backend={BACKEND} n={N} =====")

    run_logged(
        f"full_additional_artifacts_{safe_name}_n{N}",
        f"""
        PYTHONPATH=src bash scripts/runs/run_additional_paper_artifacts.sh \
          --cells {CELL} \
          --n-examples {N} \
          --seed 0 \
          {backend_arg(BACKEND)}
        """,
        tail=80,
    )

## 12 - EXECUTE: core R1-R5 on the 2 allowlist cells
The standard matrix experiments (separate from additional artifacts).


In [ ]:
N = 30

CORE_EXPS = [
    "r1_checkability",
    "r2_redundancy",
    "r3_responsibility",
    "r4_risk_privacy",
    "r5_overhead",
]

def exp_script(exp: str) -> str:
    return f"scripts/experiments/run_{exp}.py"

for CELL in ALLOWLIST:
    ds, model = CELL.split(":", 1)
    BACKEND = pcg_backend_for(CELL)

    for exp in CORE_EXPS:
        script = exp_script(exp)
        supports_backend = script_supports_arg(script, "--backend")
        backend_part = f"--backend {BACKEND}" if supports_backend else ""

        extra = "--k-values 1 2 4" if exp == "r2_redundancy" else ""
        safe_name = f"{exp}_{CELL}".replace(":", "__").replace("/", "_")

        print(f"\n=== {exp}: {CELL} backend={BACKEND} supports_backend={supports_backend} ===")

        run_logged(
            f"core_{safe_name}_n{N}",
            f"""
            PYTHONPATH=src python {script} \
              --dataset {ds} \
              --model {model} \
              {backend_part} \
              --n-examples {N} \
              --seeds 0 \
              {extra}
            """,
            tail=80,
        )

## 13 - EXECUTE: SOTA baselines on the 2 allowlist cells
5 twins via --pairs/--backend-mode; ShieldAgent separately via --cell + Anthropic.


In [ ]:
N = 30

SOTA_METHODS = ["agentrr", "verimap", "atlasprism", "pcnrec", "clbc"]

LLAMA_PAIRS = ",".join([c for c in ALLOWLIST if not c.endswith(":deepseek-v3")])
DEEPSEEK_PAIRS = ",".join([c for c in ALLOWLIST if c.endswith(":deepseek-v3")])

for method in SOTA_METHODS:
    script = f"scripts/baselines/{method}/run_{method}_r1_r5.py"
    script_path = LOCAL_REPO / script

    if not script_path.exists():
        print(f"SKIP {method}: missing {script}")
        continue

    help_text = subprocess.run(
        ["bash", "-lc", f"cd {LOCAL_REPO} && python {script} --help"],
        capture_output=True,
        text=True,
        timeout=60,
    )
    h = help_text.stdout + help_text.stderr

    if LLAMA_PAIRS:
        print(f"\n===== SOTA {method}: {LLAMA_PAIRS} backend-mode=hf_local =====")
        run_logged(
            f"sota_{method}_llama_n{N}",
            f"""
            PYTHONPATH=src python {script} \
              --pairs {LLAMA_PAIRS} \
              --seeds 0 \
              --n-examples {N} \
              --backend-mode hf_local
            """,
            tail=80,
        )

    if DEEPSEEK_PAIRS:
        if "deepseek" in h:
            print(f"\n===== SOTA {method}: {DEEPSEEK_PAIRS} backend-mode=deepseek =====")
            run_logged(
                f"sota_{method}_deepseek_n{N}",
                f"""
                PYTHONPATH=src python {script} \
                  --pairs {DEEPSEEK_PAIRS} \
                  --seeds 0 \
                  --n-examples {N} \
                  --backend-mode deepseek
                """,
                tail=80,
            )
        else:
            print(f"SKIP {method} DeepSeek pair: script does not advertise deepseek backend-mode")


### ShieldAgent (Anthropic-backed, separate CLI)
Requires ANTHROPIC_API_KEY. Runs per-cell with explicit jsonl paths.


In [ ]:
import os
from pathlib import Path

N = 30

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("SKIP ShieldAgent: ANTHROPIC_API_KEY not set")
else:
    for CELL in ALLOWLIST:
        slug = CELL.replace(":", "__").replace("/", "_")
        base = OUT_ROOT / "baselines" / "shieldagent" / slug
        base.mkdir(parents=True, exist_ok=True)

        script = "scripts/baselines/shieldagent/run_shieldagent_r1_r5_comparative.py"
        if not (LOCAL_REPO / script).exists():
            print(f"SKIP ShieldAgent: missing {script}")
            continue

        print(f"\n===== ShieldAgent: {CELL} =====")
        run_logged(
            f"shieldagent_{slug}_n{N}",
            f"""
            PYTHONPATH=src python {script} \
              --cell {CELL} \
              --input-jsonl {base}/input.jsonl \
              --output-jsonl {base}/output.jsonl \
              --metrics-json {base}/metrics.json \
              --r2-json {base}/r2.json \
              --r3-json {base}/r3.json \
              --r4-json {base}/r4.json
            """,
            tail=80,
        )

## 14 - Zip results for local sync-back
Download the zip, unzip into local repo `results/`, then rebuild figures/tables locally.


In [ ]:
import time
from pathlib import Path
import subprocess

STAMP = time.strftime("%Y%m%d-%H%M%S")
ZIP = ZIP_ROOT / f"pcg_frontier_results_{STAMP}.zip"

print("RUN_ROOT:", RUN_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("ZIP:", ZIP)

subprocess.check_call([
    "bash", "-lc",
    f"cd {RUN_ROOT} && zip -r -q {ZIP} results"
])

print("wrote:", ZIP)
print("Sync-back: download this zip, unzip into local repo root so it merges results/...")

In [ ]:
from google.colab import files
files.download(str(ZIP))